# Week 2 — Day 2: Structured Outputs
### CalderR Agentic AI Engineering Internship · Tuesday 30 June 2026

**Today's stack:** Python · Pydantic v2 (`2.9.2`) · LangChain (`langchain-groq`) · Groq API (`openai/gpt-oss-120b`)

**Scope (from the Week 2 brief):**
- Theory: Pydantic v2 validators, `model_config`, field aliases; JSON mode vs function-calling-based structured output; output parsers
- Practice: build 5 complex Pydantic models for AI output schemas
- Applied: implement structured extraction with LangChain's `with_structured_output()`; parse messy LLM output into clean Python objects
- Lab 2.1: job-posting extractor (title, company, salary range, required skills, location, remote status) as a validated Pydantic model

**Continuity note from yesterday:** Day 1 used raw text plus a regex (`extract_final_answer`) to scrape a numeric answer out of free-form CoT output. That regex was fragile by design -- a placeholder for exactly what today replaces it with. From here on, the model's output is a typed, validated object, not a string you hope has the right shape.

**A live compatibility issue worth knowing before you start:** there are two different LangChain code paths that touch `openai/gpt-oss-120b` and structured output, and they behave differently:
- `ChatGroq(...).with_structured_output(YourModel)` -- **works**. It uses Groq's native `json_schema` response format under the hood, and Groq's own docs confirm `strict` mode is explicitly supported for `openai/gpt-oss-20b` and `openai/gpt-oss-120b`. This is what today's notebook uses.
- `langchain.agents.create_agent(model=..., response_format=YourModel)` -- **currently broken** on this model (tracked as `langchain-ai/langchain#34155`, filed Dec 2025, still open as of this writing). The default `ProviderStrategy` sends `strict=True` in a way the Groq SDK itself rejects for this code path, and the `ToolStrategy` fallback causes the model to ignore the schema and emit free text instead. You won't hit this today -- `create_agent` isn't introduced until later -- but when you do, expect it, and don't burn an hour assuming your Pydantic model is the problem.

---

## 0. Environment Setup

Pin these -- `langchain-groq` is a fast-moving integration package and minor version bumps have changed default `with_structured_output()` behavior before (see the GitHub issue referenced above for an example of exactly that kind of regression).

In [1]:
# requirements-day2.txt
# pydantic==2.9.2
# langchain-core==0.3.51
# langchain-groq==0.2.3
# groq==0.32.0
# python-dotenv==1.0.1

!pip install -q --upgrade pydantic langchain-core langchain-groq groq python-dotenv



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
from datetime import date
from decimal import Decimal
from enum import Enum
from typing import Literal

from dotenv import load_dotenv
from pydantic import (
    BaseModel, # refers to the base class for creating data models
    Field, # refers to a function used to define model fields with additional metadata
    ConfigDict, # refers to a class used to configure model behavior
    field_validator, # refers to a decorator for defining custom validation logic for model fields
    model_validator, # refers to a decorator for defining custom validation logic for the entire model
    AliasChoices, # refers to a class used to define alias choices for model fields
    ValidationError, # refers to an exception class raised when validation errors occur
)
from langchain_groq import ChatGroq

load_dotenv()

API_KEY = os.environ.get("GROQ_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "GROQ_API_KEY not found. Create a `.env` file next to this notebook "
        "with GROQ_API_KEY=gsk_... and keep `.env` out of git."
    )

MODEL_NAME = "openai/gpt-oss-120b"

llm = ChatGroq(model=MODEL_NAME, api_key=API_KEY, temperature=0)

print(f"LangChain ChatGroq client ready. Model: {MODEL_NAME}")


LangChain ChatGroq client ready. Model: openai/gpt-oss-120b


## 1. Why Structured Output Matters for Production AI Systems

### 1.1 The problem with free text

An LLM's native output is a string. The moment any downstream code needs a `salary_min` to compare against a filter, or a `severity` to route a ticket, you're parsing that string -- and string-parsing an inherently non-deterministic generator is the failure mode Day 1's `extract_final_answer` regex demonstrated on purpose. Three concrete ways this breaks in production, not just in theory:

1. **Format drift across calls.** The same prompt, run twice, can phrase a number as `"$85,000"`, `"85k"`, or `"USD 85,000/year"`. A regex tuned for one format silently fails (returns `None` or the wrong match) on the others -- and "silently" is the dangerous part: no exception, no traceback, just a wrong or missing value flowing downstream. This is the same bug class as your LeakWatch project's *silent false-negative on rules-load failure* -- the system looks like it ran fine, but the output is wrong.
2. **Type confusion.** `"required_skills": "Python, SQL, Docker"` (a string) vs `["Python", "SQL", "Docker"]` (a list) -- both are plausible completions for the same prompt depending on phrasing nuance, sampling temperature, and the model's mood that token. Downstream code that does `for skill in result["required_skills"]` behaves completely differently (iterates characters of a string vs. items of a list) depending on which the model happened to produce.
3. **No fail-fast boundary.** Without validation, a malformed field doesn't throw where the malformation occurred -- it propagates until something *else* breaks, often several function calls downstream, by which point the actual cause is gone from the stack trace.

### 1.2 What structured output actually buys you

- **A schema contract** -- you declare the shape once; every response is checked against it before your code ever touches the data.
- **Fail-fast, localized errors** -- a `ValidationError` raised at the parse boundary tells you exactly which field violated which constraint, immediately, not three function calls later.
- **Type safety for the rest of your pipeline** -- once validated, a `JobPosting` object behaves like any other typed Python object; no defensive `isinstance` checks scattered through downstream code.
- **Self-documenting schemas** -- the Pydantic model *is* the contract; a teammate (or future you) reads the model definition instead of reverse-engineering what fields a prompt happens to produce.

### 1.3 Two distinct mechanisms, often conflated

| | JSON Mode | Function/Tool Calling for structured output |
|---|---|---|
| What you set | `response_format={"type": "json_object"}` | `tools=[...]` + the model emits a `tool_call` |
| Guarantee | Output **is valid JSON** -- no guarantee it matches *your* schema's fields/types | Output matches the named function's parameter schema (when the model behaves) |
| Failure mode | Valid JSON, wrong shape (missing fields, extra fields, wrong types) | Model ignores the tool and replies with prose instead (a real, documented failure mode on Groq's forum and in the LangChain issue above for `gpt-oss-120b` under certain configurations) |
| Where Groq's native **Structured Outputs** fits | `response_format={"type": "json_schema", "json_schema": {...}, "strict": true}` -- newer than plain JSON mode, **does** enforce your schema via constrained decoding when `strict=True` is supported (it is, for `gpt-oss-20b`/`120b`) | -- |

`with_structured_output()`, which we use for the rest of this notebook, abstracts over these and picks Groq's `json_schema` strict-mode path by default for models that support it -- which is why it works for the model we're using today, and why the *separate* `create_agent` code path mentioned above (which routes through a different internal strategy) doesn't.

---

## 2. Pydantic v2 Core Concepts

### 2.1 The v1 to v2 migration, briefly (you will see v1 syntax in older tutorials)

If you copy a Pydantic snippet from a 2022-era blog post, expect it to be wrong in v2. The headline changes:

| v1 (deprecated/removed) | v2 (current) |
|---|---|
| `@validator` | `@field_validator` (requires `@classmethod`, explicit `mode="before"/"after"`) |
| `@root_validator` | `@model_validator(mode="before"/"after")` |
| `class Config:` inner class | `model_config = ConfigDict(...)` class attribute |
| `.dict()` | `.model_dump()` |
| `.json()` | `.model_dump_json()` |
| `.parse_obj(...)` | `.model_validate(...)` |
| `Field(alias=...)` only | `Field(validation_alias=..., serialization_alias=...)` -- split, plus `AliasChoices`/`AliasPath` for multi-source aliasing |

`pydantic.v1` is still importable as a compatibility shim for incremental migration, but write new code in v2 syntax -- there's no reason to write v1-style validators in a project started in 2026.

### 2.2 Field validators -- `@field_validator`

Runs on a single field. `mode="after"` (the default) validates *after* Pydantic's own type coercion; `mode="before"` runs on the raw input before type coercion, which matters when you need to clean a value (e.g., strip a currency symbol) before Pydantic tries to coerce it to `int` and fails.

In [ ]:
class SalaryExample(BaseModel):
    # mode="after": value has already been coerced to int by the time the
    # validator runs. Use this when checking the COERCED value's constraints.
    salary_min: int

    @field_validator("salary_min", mode="after")
    @classmethod
    def salary_must_be_positive(cls, v: int) -> int:
        if v < 0:
            raise ValueError("salary_min cannot be negative")
        return v


class RawSalaryExample(BaseModel):
    # mode="before": value arrives as whatever the input gave us -- here we
    # strip "$" and "," BEFORE Pydantic tries (and would otherwise fail) to
    # coerce "$85,000" to an int.
    salary_min: int

    @field_validator("salary_min", mode="before")
    @classmethod
    def strip_currency_formatting(cls, v):
        if isinstance(v, str): # explain: check if the value is a string
            v = v.replace("$", "").replace(",", "").strip() # explain: remove dollar signs, commas, and whitespace from the string
        return v


# mode="after" example: catches a value that's already a valid int but
# violates a business rule.
try:
    SalaryExample(salary_min=-500)
except ValidationError as e:
    print("=== mode='after' caught a business-rule violation ===")
    print(e)

print()

# mode="before" example: cleans "$85,000" into something int() can parse,
# BEFORE Pydantic's coercion step would otherwise raise on the "$" character.
cleaned = RawSalaryExample(salary_min="$85,000")
print("=== mode='before' cleaned a malformed string into a valid int ===")
print(cleaned)


=== mode='after' caught a business-rule violation ===
1 validation error for SalaryExample
salary_min
  Value error, salary_min cannot be negative [type=value_error, input_value=-500, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

=== mode='before' cleaned a malformed string into a valid int ===
salary_min=85000


**Why this distinction matters in practice, not just as trivia:** an LLM extracting a salary from messy text will sometimes emit `"$85,000"` as a string even when your schema says `int`. If you only have an `after` validator, Pydantic's type coercion fails *before* your validator ever runs -- you get a generic "input should be a valid integer" error instead of your custom message, and the cleaning logic you wanted never executes. `before` validators are how you intercept and normalize messy LLM output before strict typing rejects it outright.

### 2.3 Model validators -- `@model_validator`

Runs on the whole model, after individual fields have validated -- use this for cross-field constraints that no single field's validator could express on its own (e.g., "max must be at least min").

In [4]:
class SalaryRangeExample(BaseModel):
    salary_min: int
    salary_max: int

    @model_validator(mode="after")
    def max_must_exceed_min(self) -> "SalaryRangeExample":
        if self.salary_max < self.salary_min:
            raise ValueError(
                f"salary_max ({self.salary_max}) cannot be less than "
                f"salary_min ({self.salary_min})"
            )
        return self


try:
    SalaryRangeExample(salary_min=90000, salary_max=70000)
except ValidationError as e:
    print("=== Cross-field validation caught an inverted range ===")
    print(e)


=== Cross-field validation caught an inverted range ===
1 validation error for SalaryRangeExample
  Value error, salary_max (70000) cannot be less than salary_min (90000) [type=value_error, input_value={'salary_min': 90000, 'salary_max': 70000}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


This exact bug class -- an LLM extracting a salary range and occasionally swapping or mis-ordering min/max from ambiguous phrasing like *"up to $90k, starting around $70k"* -- is precisely what a single-field validator structurally cannot catch (each number is individually valid; only the *relationship* is wrong). This is the validator pattern Lab 2.1 below will actually need.

### 2.4 `model_config` -- model-level behavior settings

Replaces v1's inner `class Config`. Three settings worth knowing for LLM-output parsing specifically:

In [7]:
class StrictExtraction(BaseModel):
    # extra="forbid": raise if the model returns a field you didn't declare.
    # This catches LLM "helpfulness drift" -- e.g. it adds a `notes` field
    # you never asked for. extra="ignore" (the v2 default) would silently
    # drop it instead; extra="allow" would silently accept it onto the model.
    # For AI-output schemas, "forbid" is usually the right default: an
    # unexpected field is a signal your prompt/schema is out of sync with
    # what the model is actually producing, and you want to know.
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

    title: str
    company: str


# extra field triggers a validation error instead of silently passing through
try:
    StrictExtraction(title="ML Engineer", company="Acme", department="R&D")
except ValidationError as e:
    print("=== extra='forbid' caught an unexpected field ===")
    print(e)

print()

# str_strip_whitespace=True auto-trims whitespace LLMs sometimes pad outputs
# with -- one less thing your prompt needs to micromanage.
clean = StrictExtraction(title="  ML Engineer  ", company="Acme ")
print("=== str_strip_whitespace auto-cleaned padding ===")
print(repr(clean))


=== extra='forbid' caught an unexpected field ===
1 validation error for StrictExtraction
department
  Extra inputs are not permitted [type=extra_forbidden, input_value='R&D', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden

=== str_strip_whitespace auto-cleaned padding ===
StrictExtraction(title='ML Engineer', company='Acme')


### 2.5 Field aliases -- `AliasChoices` / `AliasPath`

Useful when the model (or an upstream API you're normalizing) might emit a field under one of several possible names, and you don't want five near-duplicate Pydantic models just to handle naming variance.

In [15]:
class FlexibleJobPosting(BaseModel):
    # AliasChoices: accept ANY of these key names from the input data while
    # always exposing the same canonical attribute name (`job_title`) to the
    # rest of your code. Useful when extraction prompts produce inconsistent
    # key naming across runs/models, or when normalizing data from multiple
    # upstream sources that don't share a naming convention.
    job_title: str = Field(validation_alias=AliasChoices("job_title", "title", "position"))


# All three of these represent the SAME logical input under different key names.
print(FlexibleJobPosting.model_validate({"job_title": "Backend Engineer"}))
print(FlexibleJobPosting.model_validate({"title": "Backend Engineer"}))
print(FlexibleJobPosting.model_validate({"position": "Backend Engineer"}))


job_title='Backend Engineer'
job_title='Backend Engineer'
job_title='Backend Engineer'


In [13]:
from pydantic import BaseModel, model_validator

class FlexibleJobPosting(BaseModel):
    # Ab hum yahan koi Field ya AliasChoices use nahi kar rahe.
    # Canonical attribute sirf `job_title` hi rahega.
    job_title: str

    @model_validator(mode="before")
    @classmethod
    def handle_title_aliases(cls, data):
        # 1. Check karein ke incoming data ek dictionary hai ya nahi
        if isinstance(data, dict):
            # 2. Mumkina keys ki list jinhein hum accept karna chahte hain
            aliases = ["job_title", "title", "position"]
            
            # 3. Loop chala kar check karein ke data mein in mein se koi key maujood hai?
            for alias in aliases:
                if alias in data:
                    # Agar "title" ya "position" milti hai, toh uski value ko 
                    # canonical key `job_title` mein copy kar dein.
                    data["job_title"] = data[alias]
                    break  # Pehli match hone wali key milte hi loop rok dein

        # 4. Cleaned data ko return karein taake Pydantic baqi validation kar sake
        return data


# Testing: Teeno tareeqon se bilkul pehle jesa hi output aayega
print(FlexibleJobPosting.model_validate({"job_title": "Backend Engineer"}))
print(FlexibleJobPosting.model_validate({"title": "Backend Engineer"}))
print(FlexibleJobPosting.model_validate({"position": "Backend Engineer"}))

job_title='Backend Engineer'
job_title='Backend Engineer'
job_title='Backend Engineer'


**Caution on overusing this for LLM output specifically:** `AliasChoices` is a good fit for normalizing *external* data sources with genuinely inconsistent naming (e.g., three different upstream APIs). It is a weaker fix for *LLM* output naming inconsistency -- if your own prompt is producing `title` sometimes and `job_title` other times, the more robust fix is tightening the prompt/schema description, not adding aliases to absorb prompt-level inconsistency you control. Reach for aliases when the inconsistency is structural (multiple real sources), not when it's just your own prompt being underspecified.

---

## 3. Five Complex Pydantic Models for AI Output Schemas

Per the brief's requirement to "build 5 complex Pydantic models for AI output schemas." Each one below targets a distinct real extraction/classification task and deliberately exercises a different combination of today's concepts (nested models, enums, validators, aliases, `model_config`) so this section also doubles as a reference you can copy from later in the internship.

### 3.1 Model 1 -- Job Posting Extractor (this is also Lab 2.1's schema)

Nested model (`SalaryRange`), an `Enum` for a constrained category, a `mode="before"` field validator for currency cleanup, and a `mode="after"` model validator for cross-field range checking.

In [16]:
class RemoteStatus(str, Enum):
    REMOTE = "remote"
    HYBRID = "hybrid"
    ONSITE = "onsite"
    UNSPECIFIED = "unspecified"


class SalaryRange(BaseModel):
    model_config = ConfigDict(extra="forbid")

    salary_min: int | None = Field(default=None, description="Lower bound of the salary range, in the stated currency's smallest common unit (e.g. whole dollars, not cents)")
    salary_max: int | None = Field(default=None, description="Upper bound of the salary range")
    currency: str = Field(default="USD", description="ISO 4217 currency code, e.g. USD, PKR, EUR")

    # Field validator to strip currency symbols and formatting from salary inputs
    @field_validator("salary_min", "salary_max", mode="before")
    @classmethod 
    def strip_currency_symbols(cls, v):
        if isinstance(v, str):
            v = v.replace("$", "").replace(",", "").replace("k", "000").strip()
            return int(v) if v else None
        return v

    # Model validator to ensure salary_max is not below salary_min
    @model_validator(mode="after")
    def max_not_below_min(self) -> "SalaryRange":
        if self.salary_min is not None and self.salary_max is not None:
            if self.salary_max < self.salary_min:
                raise ValueError(
                    f"salary_max ({self.salary_max}) is below salary_min ({self.salary_min})"
                )
        return self


class JobPosting(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

    title: str = Field(description="Job title as stated in the posting")
    company: str = Field(description="Hiring company name")
    salary_range: SalaryRange | None = Field(default=None, description="Salary range if mentioned, otherwise null")
    required_skills: list[str] = Field(default_factory=list, description="Technical skills explicitly required, not 'nice to have'")
    location: str | None = Field(default=None, description="City/region if stated, otherwise null")
    remote_status: RemoteStatus = Field(default=RemoteStatus.UNSPECIFIED)

    @field_validator("required_skills", mode="before")
    @classmethod
    def split_if_comma_string(cls, v):
        # Defensive normalization: if the model emits a comma-joined string
        # instead of a list (a real, observed LLM output variance), split it
        # rather than letting Pydantic reject the whole field.
        if isinstance(v, str):
            return [s.strip() for s in v.split(",") if s.strip()]
        return v


# Sanity check with deliberately messy input mimicking realistic LLM output variance
test = JobPosting.model_validate({
    "title": "  Senior Backend Engineer  ",
    "company": "Initech",
    "salary_range": {"salary_min": "$90,000", "salary_max": "$120,000", "currency": "USD"},
    "required_skills": "Python, PostgreSQL, Docker",  # comma string, not a list
    "location": "Remote (US timezones)",
    "remote_status": "remote",
})
print(test.model_dump_json(indent=2))


{
  "title": "Senior Backend Engineer",
  "company": "Initech",
  "salary_range": {
    "salary_min": 90000,
    "salary_max": 120000,
    "currency": "USD"
  },
  "required_skills": [
    "Python",
    "PostgreSQL",
    "Docker"
  ],
  "location": "Remote (US timezones)",
  "remote_status": "remote"
}


### 3.2 Model 2 -- Code Review Finding (severity-tiered structured report)

`Literal` for a closed severity enum (lighter-weight than a full `Enum` class when you don't need methods on the type), and a validator that enforces a business rule tying severity to required fields -- a CRITICAL finding without a `cwe_id` is plausible from an LLM but a quality gap you want caught at the schema boundary, not three steps downstream in a report generator.

In [17]:
class CodeFinding(BaseModel):
    model_config = ConfigDict(extra="forbid")

    file_path: str
    line_number: int = Field(gt=0, description="1-indexed line number; must be positive")
    severity: Literal["LOW", "MEDIUM", "HIGH", "CRITICAL"]
    category: Literal["bug", "security", "style", "performance"]
    description: str = Field(min_length=10, description="Human-readable explanation, at least 10 chars -- rejects lazy one-word findings")
    suggested_fix: str
    cwe_id: str | None = Field(default=None, description="CWE identifier if category is 'security', e.g. 'CWE-89'")

    @model_validator(mode="after")
    def security_findings_should_have_cwe(self) -> "CodeFinding":
        if self.category == "security" and self.severity in ("HIGH", "CRITICAL") and not self.cwe_id:
            raise ValueError(
                "HIGH/CRITICAL security findings must include a cwe_id -- "
                "a severity claim without a CWE reference is under-specified "
                "for a production audit report"
            )
        return self


class CodeReviewReport(BaseModel):
    model_config = ConfigDict(extra="forbid")

    findings: list[CodeFinding]
    files_reviewed: int = Field(gt=0)

    @field_validator("findings")
    @classmethod
    def must_have_at_least_one_finding_or_be_explicit(cls, v):
        # Not a hard error -- an empty findings list on a real review is
        # suspicious (did the reviewer actually run, or silently no-op?) but
        # not necessarily wrong, so we warn via print rather than raise.
        # This mirrors the LeakWatch bug class: "ran with no findings" needs
        # to be distinguishable from "ran and found nothing."
        if len(v) == 0:
            print("WARNING: zero findings reported -- verify the reviewer actually executed, didn't silently no-op")
        return v


try:
    CodeFinding(
        file_path="api/auth.py",
        line_number=42,
        severity="CRITICAL",
        category="security",
        description="SQL query built via f-string concatenation",
        suggested_fix="Use a parameterized query",
        cwe_id=None,  # missing -- should fail
    )
except ValidationError as e:
    print("=== Caught a CRITICAL security finding missing a CWE id ===")
    print(e)


=== Caught a CRITICAL security finding missing a CWE id ===
1 validation error for CodeFinding
  Value error, HIGH/CRITICAL security findings must include a cwe_id -- a severity claim without a CWE reference is under-specified for a production audit report [type=value_error, input_value={'file_path': 'api/auth.p... query', 'cwe_id': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


### 3.3 Model 3 -- Sentiment Classification with Confidence (numeric constraints + cross-field semantic check)

`Field(ge=..., le=...)` for bounded numeric ranges, plus a validator checking that confidence and label are *internally consistent* -- an LLM can produce a syntactically valid but semantically incoherent combination (e.g., `"positive"` at `0.51` confidence), which type checking alone never catches.

In [20]:
class SentimentResult(BaseModel):
    model_config = ConfigDict(extra="forbid")

    text_snippet: str = Field(max_length=200, description="The text being classified, truncated for the report")
    label: Literal["positive", "negative", "neutral"]
    confidence: float = Field(ge=0.0, le=1.0, description="Model's confidence, 0.0-1.0")
    key_phrases: list[str] = Field(default_factory=list, max_length=5, description="Up to 5 phrases driving the classification")

    @model_validator(mode="after")
    def confidence_should_be_decisive_for_polar_labels(self) -> "SentimentResult":
        # A "positive" or "negative" call made at near-coin-flip confidence
        # is a signal the label itself is shaky -- in a real pipeline this
        # is exactly the kind of result you'd route to human review instead
        # of trusting outright, so we surface it as a validation error.
        if self.label in ("positive", "negative") and self.confidence < 0.55:
            raise ValueError(
                f"label='{self.label}' with confidence={self.confidence} is \n"
                f"too close to a coin flip to be a decisive polar classification \n"
                f"-- consider 'neutral' or flag for human review instead"
            )
        return self


try:
    SentimentResult(text_snippet="It's fine I guess.", label="positive", confidence=0.51, key_phrases=["fine"])
except ValidationError as e:
    print("=== Caught a low-confidence polar label that should be 'neutral' or flagged ===")
    print(e)


=== Caught a low-confidence polar label that should be 'neutral' or flagged ===
1 validation error for SentimentResult
  Value error, label='positive' with confidence=0.51 is 
too close to a coin flip to be a decisive polar classification 
-- consider 'neutral' or flag for human review instead [type=value_error, input_value={'text_snippet': "It's fi...'key_phrases': ['fine']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


### 3.4 Model 4 -- Multi-Step Tool-Call Plan (structural preview of tomorrow's topic)

This previews tomorrow's tool-calling material structurally, without actually calling any tools yet -- the model here just describes a *plan*. Demonstrates `list[NestedModel]` for ordered steps and a validator enforcing that step dependencies only reference earlier steps (no forward references, which would describe an impossible execution order).

In [21]:
class PlannedToolCall(BaseModel):
    model_config = ConfigDict(extra="forbid")

    step_number: int = Field(gt=0)
    tool_name: str
    tool_args: dict = Field(default_factory=dict)
    depends_on_steps: list[int] = Field(default_factory=list, description="step_numbers this step needs output from, if any")


class ToolCallPlan(BaseModel):
    model_config = ConfigDict(extra="forbid")

    goal: str
    steps: list[PlannedToolCall]

    @model_validator(mode="after")
    def dependencies_must_be_earlier_steps(self) -> "ToolCallPlan":
        step_numbers = {s.step_number for s in self.steps}
        for step in self.steps:
            for dep in step.depends_on_steps:
                if dep not in step_numbers:
                    raise ValueError(f"Step {step.step_number} depends on step {dep}, which doesn't exist in this plan")
                if dep >= step.step_number:
                    raise ValueError(
                        f"Step {step.step_number} depends on step {dep}, but {dep} is not "
                        f"strictly earlier -- this describes an impossible execution order"
                    )
        return self


try:
    ToolCallPlan(
        goal="Get weather and convert temperature to Fahrenheit",
        steps=[
            PlannedToolCall(step_number=1, tool_name="get_weather", tool_args={"city": "Lahore"}),
            PlannedToolCall(step_number=2, tool_name="convert_temp", tool_args={}, depends_on_steps=[3]),  # forward reference -- invalid
        ],
    )
except ValidationError as e:
    print("=== Caught an impossible forward dependency in a tool-call plan ===")
    print(e)


=== Caught an impossible forward dependency in a tool-call plan ===
1 validation error for ToolCallPlan
  Value error, Step 2 depends on step 3, which doesn't exist in this plan [type=value_error, input_value={'goal': 'Get weather and... depends_on_steps=[3])]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


### 3.5 Model 5 -- Document Action-Item Extractor (dates, `Decimal` for money, alias-based normalization)

Brings together `date` parsing, `Decimal` (not `float`, for money -- see the warning below), and `AliasChoices` for a field that genuinely does vary by source-document convention (some incident docs say `owner`, others say `assignee`, others `responsible_party`).

In [22]:
class ActionItem(BaseModel):
    model_config = ConfigDict(extra="forbid")

    description: str = Field(min_length=5)
    owner: str | None = Field(
        default=None,
        validation_alias=AliasChoices("owner", "assignee", "responsible_party"),
        description="Person or role responsible",
    )
    due_date: date | None = Field(default=None, description="ISO 8601 date if a deadline is stated, otherwise null")
    estimated_cost: Decimal | None = Field(
        default=None,
        description="Estimated cost if mentioned. Use Decimal, not float -- "
                    "float is binary floating point and cannot represent "
                    "currency exactly (0.1 + 0.2 != 0.3 in float); Decimal "
                    "avoids silent rounding-error accumulation in financial figures.",
    )

    @field_validator("due_date", mode="before")
    @classmethod
    def parse_loose_date_strings(cls, v):
        if isinstance(v, str):
            from datetime import datetime
            for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%B %d, %Y", "%d %B %Y"):
                try:
                    return datetime.strptime(v, fmt).date()
                except ValueError:
                    continue
        return v


# Three different source-document conventions for the same logical field
print(ActionItem.model_validate({"description": "Patch the upload endpoint", "owner": "Amaima", "due_date": "2026-07-02"}))
print(ActionItem.model_validate({"description": "Notify affected users", "assignee": "Support team", "due_date": "2 July 2026"}))
print(ActionItem.model_validate({"description": "Budget the fix", "responsible_party": "Finance", "estimated_cost": "1500.50"}))


description='Patch the upload endpoint' owner='Amaima' due_date=datetime.date(2026, 7, 2) estimated_cost=None
description='Notify affected users' owner='Support team' due_date=datetime.date(2026, 7, 2) estimated_cost=None
description='Budget the fix' owner='Finance' due_date=None estimated_cost=Decimal('1500.50')


**Why `Decimal` over `float` for money, concretely:** `float(0.1) + float(0.2) == 0.30000000000000004` in IEEE-754 binary floating point -- not a Pydantic quirk, a hardware-level representation fact. For a single estimated-cost field that's cosmetic; for anything that *sums* many extracted costs (e.g., aggregating action-item budgets across a document), those errors compound. `Decimal` parses from a string exactly and avoids the whole class of bug. Pydantic v2 supports `Decimal` natively as a field type, including JSON Schema generation for it.

---

## 4. Structured Extraction with `with_structured_output()`

### 4.1 The pattern

`with_structured_output()` wraps a chat model so that, instead of returning an `AIMessage` with a `.content` string, invoking it returns an instance of your Pydantic model directly -- parsed and validated. Internally (for Groq + `gpt-oss-120b`) this routes through Groq's native `json_schema` strict-mode response format, which uses **constrained decoding**: the model is structurally prevented from generating tokens that would violate your schema, rather than just being asked nicely and hoping.

In [23]:
structured_llm = llm.with_structured_output(JobPosting)

messy_posting = (
    "We're hiring!! Senior Backend Eng @ Initech. This is a fully remote "
    "role (US/Canada timezones only pls). Comp is 90-120k USD depending on "
    "experience. Must know Python, Postgres, and have shipped production "
    "Docker deployments. Bonus if you know Kubernetes but not required. "
    "Based out of our Austin HQ but remote-first."
)

result = structured_llm.invoke(
    f"Extract the job posting details from this text:\n\n{messy_posting}"
)

print(type(result))
print(result.model_dump_json(indent=2))


<class '__main__.JobPosting'>
{
  "title": "Senior Backend Engineer",
  "company": "Initech",
  "salary_range": {
    "salary_min": 90000,
    "salary_max": 120000,
    "currency": "USD"
  },
  "required_skills": [
    "Python",
    "Postgres",
    "Docker"
  ],
  "location": null,
  "remote_status": "remote"
}


**What just happened that's different from Day 1:** no regex, no manual JSON parsing, no `try/except json.JSONDecodeError`. `result` is already a real `JobPosting` instance -- `result.salary_range.salary_min` is an `int` you can do arithmetic on directly, `result.required_skills` is genuinely a `list[str]`. If the model's raw output had violated the schema (wrong type, missing required field, or -- thanks to our validators -- an inverted salary range), you'd get a `ValidationError` raised right here at the call site, not a malformed object silently handed downstream.

### 4.2 Parsing genuinely messy input -- the brief's "messy LLM outputs" requirement

The applied-practice line in the brief specifically says *"parse messy LLM outputs into clean Python objects."* The example above was messy in its *source text* but the extraction itself went smoothly. Let's stress-test with input that's adversarially ambiguous in ways that test the validators we wrote, not just the happy path.

In [26]:
ambiguous_posting = (
    "Looking for someone for our DevOps role. Salary: negotiable, but we "
    "paid the last guy around 150k and want to pay less this time, maybe "
    "110-13k. Hybrid - 2 days in office in our Lahore location, rest WFH. "
    "Skills needed: must know AWS, Terraform is a huge plus, Python "
    "scripting ability expected. Zaffire"
)

result2 = structured_llm.invoke(
    f"Extract the job posting details from this text:\n\n{ambiguous_posting}"
)
print(result2.model_dump_json(indent=2))


{
  "title": "DevOps Engineer",
  "company": "Zaffire",
  "salary_range": {
    "salary_min": 110000,
    "salary_max": 130000,
    "currency": "USD"
  },
  "required_skills": [
    "AWS",
    "Python"
  ],
  "location": "Lahore",
  "remote_status": "hybrid"
}


**Things to actually check against this output, not just admire that it ran:**
- Did `remote_status` correctly resolve to `"hybrid"` -- the text says "2 days in office, rest WFH," which is a paraphrase, not the literal word "hybrid"?
- Did `required_skills` include `"Terraform"`? The text frames it as *"a huge plus"* -- arguably a nice-to-have, not a hard requirement. Our model has no concept of "required vs. nice-to-have" beyond what the field's description tells the LLM to apply -- if you need that distinction reliably, it has to be a separate field (`required_skills` vs `preferred_skills`), not something you hope the LLM infers correctly from one field's name every time.
- Did the salary range pick up `110-130k`, or did the `"around 150k"` distractor leak into either bound? This is exactly the kind of thing self-consistency (Day 1, Section 3) could help validate for high-stakes extractions -- sample 3-5 times and check whether the salary range is stable across samples, instead of trusting one extraction.

## 5. Lab 2.1 -- Structured Output Extractor (brief requirement)

**Brief text:** *"Build a tool that takes unstructured job postings and extracts: title, company, salary range, required skills, location, and remote status, all as a validated Pydantic model."*

The `JobPosting` model from Section 3.1 already satisfies every field this lab asks for -- that overlap is intentional; Section 3's models were built to double as this lab's deliverable rather than as disconnected theory examples. What's left is wiring it into a small reusable extraction function and running it across a batch, which is what production "a tool that does X" actually means (not a one-off `invoke()` call).

In [27]:
def extract_job_posting(raw_text: str) -> JobPosting | None:
    """Extracts and validates a JobPosting from raw, unstructured text.
    Returns None (and prints the validation error) on failure rather than
    raising -- in a batch-processing tool, one bad input shouldn't crash
    the whole run. This mirrors the graceful-degradation principle Day 1
    previewed, which Thursday's error-handling session covers in full.
    """
    try:
        return structured_llm.invoke(
            f"Extract the job posting details from this text. If a field "
            f"isn't mentioned, leave it null/empty rather than guessing.\n\n"
            f"{raw_text}"
        )
    except ValidationError as e:
        print(f"VALIDATION FAILED for input: {raw_text[:60]}...")
        print(e)
        return None


In [28]:
test_postings = [
    (
        "Frontend Developer at PixelWorks. Remote OK. $70k-$90k. Need "
        "React, TypeScript, and CSS skills. Office is in Karachi if you "
        "want to come in."
    ),
    (
        "URGENT HIRING: Data Scientist, TechNova Inc. Onsite only, our "
        "Islamabad office. Pandas, scikit-learn, SQL required. Pay band "
        "150,000-200,000 PKR/month."
    ),
    (
        "Join our team as a DevOps Engineer! CloudBase Systems. Hybrid "
        "setup, 3 days office in Lahore. Looking for Docker, Kubernetes, "
        "AWS experience. Compensation is competitive (we don't disclose "
        "ranges publicly)."
    ),
]

batch_results = []
for posting in test_postings:
    result = extract_job_posting(posting)
    batch_results.append(result)
    if result:
        print(f"OK  | {result.title} @ {result.company} | "
              f"{result.remote_status.value} | skills={result.required_skills}")
    else:
        print("FAILED to extract")


OK  | Frontend Developer @ PixelWorks | remote | skills=['React', 'TypeScript', 'CSS']
OK  | Data Scientist @ TechNova Inc. | onsite | skills=['Pandas', 'scikit-learn', 'SQL']
OK  | DevOps Engineer @ CloudBase Systems | hybrid | skills=['Docker', 'Kubernetes', 'AWS']


**Validation report requirement (brief: "validation report"):** below is a minimal pass/fail summary across the batch -- for a 3-item demo batch this is intentionally small, but the pattern (count successes, list failures with their input snippet) is what scales to a real batch of hundreds of postings without modification.

In [29]:
def validation_report(results: list, inputs: list) -> None:
    total = len(results)
    succeeded = sum(1 for r in results if r is not None)
    print(f"Validation Report: {succeeded}/{total} postings extracted successfully ({succeeded/total:.0%})")
    print()
    for i, (r, raw) in enumerate(zip(results, inputs), 1):
        status = "PASS" if r is not None else "FAIL"
        print(f"[{i}] {status} -- {raw.strip()[:50]}...")
        if r is not None:
            missing = [
                field for field in ("title", "company", "location")
                if getattr(r, field) is None
            ]
            if missing:
                print(f"      note: extracted but missing optional fields: {missing}")

validation_report(batch_results, test_postings)


Validation Report: 3/3 postings extracted successfully (100%)

[1] PASS -- Frontend Developer at PixelWorks. Remote OK. $70k-...
[2] PASS -- URGENT HIRING: Data Scientist, TechNova Inc. Onsit...
[3] PASS -- Join our team as a DevOps Engineer! CloudBase Syst...


### 5.1 Note for the third test posting

The third posting explicitly states *"we don't disclose ranges publicly"* -- a case where the *absence* of salary information is itself a meaningful, stated fact, not a missing-data gap to fill. Check whether `salary_range` came back as `None` (correct: nothing to extract) versus the model hallucinating a plausible-sounding range anyway despite the text explicitly saying it won't disclose one. This is a real, observed LLM failure mode -- models sometimes fill in "typical" values for a field type even when the source text actively states the information isn't available -- and it's exactly the kind of thing schema validation *cannot* catch (a hallucinated `90000` is a perfectly valid `int`) but a human spot-check or a stricter prompt instruction can.

---

## 6. Self-Check Before Wednesday

1. Why does `extra="forbid"` matter more for AI-output schemas specifically than for, say, a schema validating your own internal API's request bodies?
2. Walk through what happens, step by step, if you call `JobPosting.model_validate()` on a dict where `required_skills` is the string `"Python and SQL"` (no comma). Does Section 3.1's `split_if_comma_string` validator save you here? Why or why not?
3. You now have two different "make the model return X" mechanisms in your toolkit: prompting the model to emit JSON manually (parsed with `json.loads` + `model_validate`) versus `with_structured_output()`'s constrained decoding. Name one situation where you'd still prefer the manual route despite the extra parsing risk.

Tomorrow (Wednesday) is tool/function calling -- binding actual callable Python functions to a model via LangChain's `@tool` decorator and `BaseTool`/`StructuredTool`, so the model can *act*, not just structure its description of what it found. Today's Pydantic models are the direct foundation for tomorrow's tool *argument* schemas -- a tool's parameters are validated exactly the same way a `JobPosting` extraction was here.
